In [ ]:
import base64
import pandas as pd
import os

from sqlalchemy import create_engine


# --- CONFIGURACIÓN DE POSTGRES ---
DB_USER = os.getenv("DB_USER", "postgres")
DB_PASS = os.getenv("DB_PASS", "postgres")
DB_HOST = os.getenv("DB_HOST", "192.168.100.50")
DB_PORT = os.getenv("DB_PORT", "5433")
DB_NAME = os.getenv("DB_NAME", "ingresos_db")

tabla = "ingresos"  # Nombre de la tabla en PostgreSQL

DATABASE_URL = f"postgresql://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

engine = create_engine(
    DATABASE_URL, 
    connect_args={'client_encoding': 'utf8'}
)
   

def limpiar_unicode(df):
    df = df.replace({'\u2010': '-', '\u2013': '-', '\u2014': '-'}, regex=True)
    return df.astype(str)


df = pd.read_excel("C:\\Users\\mantenimiento\\Nextcloud\\2026\\1.Informes\\INF Control de Almacen\\Reportes Ingresos Salidas Almacen\\INGRESOS ABRIL GOM 2026.xlsx", skiprows=7, engine='openpyxl')
            
# Limpieza según tu lógica
if 'Código Ingreso' in df.columns:
    df = df[df['Código Ingreso'].notna()]

columnas_ui_nombres = ['Nº', 'Sistema', 'Tipo Ingreso', 'Fecha Ingreso', 'Código Ingreso', 'Estado', 'Motivo de Ingreso', 'Gerencia Solicitante', 'Departamento Solicitante', 'Ubicación', 'Tipo de Gasto', 'Partida Presupuestaria','Proveedor', 'Código Item', 'Código Anterior Item', 'Item', 'Unidad Medida', 'Grupo', 'Subgrupo', 'Cantidad', 'Precio Unitario 100%', 'Importe (Histórico) 100%', 'CF-IVA', 'Importe Calculado']
columnas_db = ['id', 'sistema', 'tipo', 'fecha_ingreso', 'codigo_ingreso', 'estado', 'motivo', 'gerencia', 'departamento', 'ubicacion', 'gasto', 'partida', 'proveedor', 'codigo', 'codigo_anterior', 'item', 'unidad', 'grupo', 'subgrupo', 'cantidad', 'punitario', 'total', 'impuesto', 'calculado']

mapping = dict(zip(columnas_ui_nombres, columnas_db))
df = df.rename(columns=mapping)

# Solo enviamos a la BD las columnas mapeadas que existen en el DF
cols_to_keep = [c for c in columnas_db if c in df.columns]
df = df[cols_to_keep]

df = limpiar_unicode(df)

# El id del excel se ignora para que Postgres use el autoincremental (id_auto)
if 'id' in df.columns:
    df = df.drop(columns=['id'])

# Asegurar tabla (usando las columnas de la BD menos el id_auto)
cols_for_sql = [c for c in df.columns]


cols_ui_formatted = [{"name": i, "id": i} for i in df.columns]

df

# Insertar en SQL
df.to_sql("ingresos", con=engine, if_exists='append', index=False)       
            
            

66

In [23]:
from sqlalchemy import create_engine, text

df_upd = pd.read_excel("ActualizarSaildas.xlsx")

col_codigo = df_upd.columns[0]
col_ubicacion = df_upd.columns[1]

datos_para_sql = [
    {
        "val_codigo": str(row[col_codigo]),
        "val_ubicacion": str(row[col_ubicacion])
    }
    for _, row in df_upd.iterrows()
]
# --- CONFIGURACIÓN DE POSTGRES ---
DB_USER = os.getenv("DB_USER", "postgres")
DB_PASS = os.getenv("DB_PASS", "postgres")
DB_HOST = os.getenv("DB_HOST", "192.168.100.50")
DB_PORT = os.getenv("DB_PORT", "5433")
DB_NAME = os.getenv("DB_NAME", "ingresos_db")

tabla = "ingresos"  # Nombre de la tabla en PostgreSQL

DATABASE_URL = f"postgresql://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

engine = create_engine(
    DATABASE_URL, 
    connect_args={'client_encoding': 'utf8'}
)

# 6. Definir la consulta de actualización estructurada con parámetros de SQLAlchemy
query_update = text("""
    UPDATE salidas 
    SET ubicacion = :val_ubicacion 
    WHERE codigo_salida = :val_codigo;
""")

# 7. Ejecutar la actualización dentro de una transacción segura
try:
    with engine.begin() as conexion:
        # conexion.execute enviará toda la lista de diccionarios en un solo bloque eficiente
        resultado = conexion.execute(query_update, datos_para_sql)
    print(f"¡Actualización exitosa! Se procesaron {len(datos_para_sql)} registros.")
    
except Exception as e:
    print(f"Ocurrió un error al actualizar la base de datos: {e}")

¡Actualización exitosa! Se procesaron 55 registros.


In [ ]:
import base64
import pandas as pd
import os

from sqlalchemy import create_engine


# --- CONFIGURACIÓN DE POSTGRES ---
DB_USER = os.getenv("DB_USER", "postgres")
DB_PASS = os.getenv("DB_PASS", "postgres")
DB_HOST = os.getenv("DB_HOST", "192.168.100.50")
DB_PORT = os.getenv("DB_PORT", "5433")
DB_NAME = os.getenv("DB_NAME", "ingresos_db")

tabla = "salidas"  # Nombre de la tabla en PostgreSQL

DATABASE_URL = f"postgresql://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

engine = create_engine(
    DATABASE_URL, 
    connect_args={'client_encoding': 'utf8'}
)
   
   
   